as we want to capture all the changes for every account we wont be checking for duplicates for this dimension

In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
accounts_path = "/Volumes/coreview/landing/organizations_data/data/accounts"

In [0]:
accounts_df = spark.read.format('csv').option('header', 'true').load(accounts_path)

Adding timestamp columns

In [0]:
accounts_df = (accounts_df.withColumn("Ingestion_timestamp", F.current_timestamp())
                        .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

In [0]:
# display(accounts_df)

In [0]:
acc_table_name = "coreview.bronze.crm_accounts"
accounts_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(acc_table_name)

In [0]:
accounts_df.count()

180

if we are doing incremental ingestion then we overwrite for initial run and then keep appending

In [0]:
# acc_table_name = "coreview.bronze.crm_accounts"

# if spark.catalog.tableExists(acc_table_name):
#     accounts_df.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(acc_table_name)
# else:
#     accounts_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(acc_table_name)


In [0]:
# %%sql
# select * from coreview.bronze.crm_accounts

as we want to capture all the changes for every organisation we wont be checking for duplicates for this dimension

In [0]:
organisations_path = '/Volumes/coreview/landing/organizations_data/data/organizations/'

the file name of the json has a date in it which is teh date showing the state of the organisation so capture it

In [0]:
org_df = (
    spark.read
    .option("multiLine", "false")
    .json(organisations_path + "*.jsonl")
    .withColumn("source_file_name", F.col("_metadata.file_name"))
)

In [0]:
display(org_df)

created_at,crm_ref,license,org_id,region,status,tenant_name,source_file_name
2025-03-25T17:34:00Z,001FXqc67PSEoGvPIC,"List(List(audit), starter, 1000)",0070366f29d4baa30aecd4bf,US,active,Harbor Insurance AU,2026-02-24.jsonl
2025-03-17T00:31:00Z,001jUw1SWAw7Uxz,"List(List(backup), standard, 100)",0367b50f9264c4019b39fc28,CA,active,Highland Legal,2026-02-24.jsonl
2025-10-11T16:53:00Z,001uyFt0STZozPa,"List(List(audit, reports), starter, 10)",040f2aec3fcc9a8ea7d33a51,AU,active,Westgate Retail,2026-02-24.jsonl
2024-10-01T01:23:00Z,001miHb18TUzEoK,"List(List(audit, backup), enterprise, 500)",04a836d16bf160863057a5f6,AU,active,Starling Energy,2026-02-24.jsonl
2024-08-02T17:49:00Z,0013fRR0TINRKOB,"List(List(), starter, 500)",07455d260415f3ed1a9f7130,AU,active,Juniper Logistics,2026-02-24.jsonl
2026-01-27T00:57:00Z,0019rawhwiRcUA0,"List(List(backup, reports), starter, 2000)",075e8f602fb1775b362e67fb,US,active,Summit Health,2026-02-24.jsonl
2024-08-17T13:12:00Z,001rz3hh3w7i8y1,"List(List(audit, reports), standard, 50)",091c578e9022ba711b0b9f25,CA,active,Sterling Foods,2026-02-24.jsonl
2025-03-02T12:13:00Z,0011zZhPeWGPAkh,"List(List(audit, reports), enterprise, 50)",094cc970cd7cf30aa30ff992,CA,active,Cedar Foods,2026-02-24.jsonl
2024-02-02T02:16:00Z,001g3rn2nFZ5b4uBSA,"List(List(audit, backup), standard, 100)",0a495cb327b4d82cf8e9ebc0,EU,active,Cedar Media,2026-02-24.jsonl
2026-02-08T09:49:00Z,0015KugxFSxBaRFLDJ,"List(List(audit), starter, 500)",0c0faa81bc5d2e473d8981a7,AU,active,Westgate Finance,2026-02-24.jsonl


In [0]:
org_df.printSchema()

root
 |-- created_at: string (nullable = true)
 |-- crm_ref: string (nullable = true)
 |-- license: struct (nullable = true)
 |    |-- addons: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- plan: string (nullable = true)
 |    |-- seats: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- tenant_name: string (nullable = true)
 |-- source_file_name: string (nullable = false)



used Genie to build the below method

In [0]:
from pyspark.sql.types import StructType

def flatten_df(df):
    """Flatten nested structs into parent_child columns (arrays are left intact)."""
    while any(isinstance(df.schema[c].dataType, StructType) for c in df.columns):
        cols = []
        for c in df.columns:
            if isinstance(df.schema[c].dataType, StructType):
                for f in df.schema[c].dataType.fields:
                    cols.append(F.col(f"{c}.{f.name}").alias(f"{c}_{f.name}"))
            else:
                cols.append(F.col(c))
        df = df.select(*cols)
    return df

org_flat = flatten_df(org_df)
org_flat.printSchema()

root
 |-- created_at: string (nullable = true)
 |-- crm_ref: string (nullable = true)
 |-- license_addons: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- license_plan: string (nullable = true)
 |-- license_seats: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- tenant_name: string (nullable = true)
 |-- source_file_name: string (nullable = false)



In [0]:
display(org_flat)

created_at,crm_ref,license_addons,license_plan,license_seats,org_id,region,status,tenant_name,source_file_name
2025-03-25T17:34:00Z,001FXqc67PSEoGvPIC,List(audit),starter,1000,0070366f29d4baa30aecd4bf,US,active,Harbor Insurance AU,2026-02-24.jsonl
2025-03-17T00:31:00Z,001jUw1SWAw7Uxz,List(backup),standard,100,0367b50f9264c4019b39fc28,CA,active,Highland Legal,2026-02-24.jsonl
2025-10-11T16:53:00Z,001uyFt0STZozPa,"List(audit, reports)",starter,10,040f2aec3fcc9a8ea7d33a51,AU,active,Westgate Retail,2026-02-24.jsonl
2024-10-01T01:23:00Z,001miHb18TUzEoK,"List(audit, backup)",enterprise,500,04a836d16bf160863057a5f6,AU,active,Starling Energy,2026-02-24.jsonl
2024-08-02T17:49:00Z,0013fRR0TINRKOB,List(),starter,500,07455d260415f3ed1a9f7130,AU,active,Juniper Logistics,2026-02-24.jsonl
2026-01-27T00:57:00Z,0019rawhwiRcUA0,"List(backup, reports)",starter,2000,075e8f602fb1775b362e67fb,US,active,Summit Health,2026-02-24.jsonl
2024-08-17T13:12:00Z,001rz3hh3w7i8y1,"List(audit, reports)",standard,50,091c578e9022ba711b0b9f25,CA,active,Sterling Foods,2026-02-24.jsonl
2025-03-02T12:13:00Z,0011zZhPeWGPAkh,"List(audit, reports)",enterprise,50,094cc970cd7cf30aa30ff992,CA,active,Cedar Foods,2026-02-24.jsonl
2024-02-02T02:16:00Z,001g3rn2nFZ5b4uBSA,"List(audit, backup)",standard,100,0a495cb327b4d82cf8e9ebc0,EU,active,Cedar Media,2026-02-24.jsonl
2026-02-08T09:49:00Z,0015KugxFSxBaRFLDJ,List(audit),starter,500,0c0faa81bc5d2e473d8981a7,AU,active,Westgate Finance,2026-02-24.jsonl


In [0]:
org_flat = (org_flat.withColumn("Ingestion_timestamp", F.current_timestamp())
                        .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

if we are doing incremental ingestion then we overwrite for initial run and then keep appending

In [0]:
org_flat.printSchema()

root
 |-- created_at: string (nullable = true)
 |-- crm_ref: string (nullable = true)
 |-- license_addons: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- license_plan: string (nullable = true)
 |-- license_seats: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- status: string (nullable = true)
 |-- tenant_name: string (nullable = true)
 |-- source_file_name: string (nullable = false)
 |-- Ingestion_timestamp: timestamp (nullable = false)
 |-- Ingestion_date: date (nullable = false)



In [0]:
# org_table_name = "coreview.bronze.organizations"

# if spark.catalog.tableExists(org_table_name):
#     org_flat.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(org_table_name)
# else:
#     org_flat.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(org_table_name)


In [0]:
org_table_name = "coreview.bronze.organizations"
org_flat.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(org_table_name)

In [0]:
org_flat.count()

12125

In [0]:
# %%sql
# select * from coreview.bronze.organizations

In [0]:
signins_path = "/Volumes/coreview/landing/organizations_data/data/signins"

In [0]:
signins_df = spark.read.format('csv').option('header', 'true').load(signins_path)

In [0]:
signins_df.count()

302593

In [0]:
signins_df = (signins_df.withColumn("Ingestion_timestamp", F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
                        .withColumn("Ingestion_date", F.to_date("Ingestion_timestamp"))
)

In [0]:
# signins_table_name = "coreview.bronze.signins"

# if spark.catalog.tableExists(signins_table_name):
#     signins_df.write.format("delta").mode("append").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(signins_table_name)
# else:
#     signins_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(signins_table_name)


In [0]:
signins_table_name = "coreview.bronze.signins"
signins_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").partitionBy('Ingestion_date').saveAsTable(signins_table_name)

In [0]:
# %%sql
# select * from coreview.bronze.signins

In [0]:
accounts_df.count()

180

In [0]:
signins_df.count()

302593

In [0]:
accounts_df.printSchema()

root
 |-- account_id: string (nullable = true)
 |-- account_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- Ingestion_timestamp: string (nullable = false)
 |-- Ingestion_date: date (nullable = true)



In [0]:
%%sql
select * from coreview.bronze.crm_accounts

account_id,account_name,country,segment,Ingestion_timestamp,Ingestion_date
001FXqc67PSEoGvPIC,Harbor Insurance,GB,SMB,2026-11-18 09:21:34,2026-11-18
001PV6mIS5NmL0yUXZ,Juniper Finance,France,Enterprise,2026-11-18 09:21:34,2026-11-18
001IjM8z1QOPEY9YHQ,Lakeside Media,Australia,Mid-Market,2026-11-18 09:21:34,2026-11-18
0019SAmry7HvhlUSWU,Falcon Partners,United Kingdom,SMB,2026-11-18 09:21:34,2026-11-18
0015KugxFSxBaRFLDJ,Westgate Finance,France,Mid-Market,2026-11-18 09:21:34,2026-11-18
001gbKHEKRgzoBVIXU,Redstone Labs,Germany,Mid-Market,2026-11-18 09:21:34,2026-11-18
001cThBZRvuCrQzJEJ,Starling Clinics,United States,Enterprise,2026-11-18 09:21:34,2026-11-18
001q9d3zuPTORaJKCX,Granite Energy,Germany,Mid-Market,2026-11-18 09:21:34,2026-11-18
001ZqKnRKNABsiJLJJ,Oakridge Builders,CA,Mid-Market,2026-11-18 09:21:34,2026-11-18
001fqkdjoiLovRyMHL,Juniper Health,germany,SMB,2026-11-18 09:21:34,2026-11-18


DataFrame[account_id: string, account_name: string, country: string, segment: string, Ingestion_timestamp: string, Ingestion_date: date]

In [0]:
%%sql
select * from coreview.bronze.crm_accounts
where account_id like '001i2Ll%'

account_id,account_name,country,segment,Ingestion_timestamp,Ingestion_date
001i2LlKYto8HjaRLJ,Crescent Labs,France,Mid-Market,2026-11-18 09:21:34,2026-11-18


DataFrame[account_id: string, account_name: string, country: string, segment: string, Ingestion_timestamp: string, Ingestion_date: date]

In [0]:
%%sql
select * from coreview.bronze.organizations
order by created_at

created_at,crm_ref,license_plan,license_seats,org_id,region,status,tenant_name,license_addon,Ingestion_timestamp,Ingestion_date
2024-01-03T03:53:00Z,001i2LlKYto8Hja,enterprise,50,bb5c0c710b807ea6c3a62ff0,EU,active,Crescent Labs US,null,2026-11-18 09:21:53,2026-11-18
2024-01-03T03:53:00Z,001i2LlKYto8Hja,enterprise,50,bb5c0c710b807ea6c3a62ff0,EU,active,Crescent Labs US,null,2026-11-18 09:21:53,2026-11-18
2024-01-03T03:53:00Z,001i2LlKYto8Hja,enterprise,50,bb5c0c710b807ea6c3a62ff0,EU,active,Crescent Labs US,null,2026-11-18 09:21:53,2026-11-18
2024-01-03T03:53:00Z,001i2LlKYto8Hja,enterprise,2000,bb5c0c710b807ea6c3a62ff0,EU,active,Crescent Labs US,null,2026-11-18 09:21:53,2026-11-18
2024-01-03T03:53:00Z,001i2LlKYto8Hja,starter,50,bb5c0c710b807ea6c3a62ff0,EU,active,Crescent Labs US,null,2026-11-18 09:21:53,2026-11-18
2024-01-03T03:53:00Z,001i2LlKYto8Hja,enterprise,50,bb5c0c710b807ea6c3a62ff0,EU,active,Crescent Labs US,null,2026-11-18 09:21:53,2026-11-18
2024-01-03T03:53:00Z,001i2LlKYto8Hja,enterprise,50,bb5c0c710b807ea6c3a62ff0,EU,active,Crescent Labs US,null,2026-11-18 09:21:53,2026-11-18
2024-01-03T03:53:00Z,001i2LlKYto8Hja,enterprise,50,bb5c0c710b807ea6c3a62ff0,EU,active,Crescent Labs US,null,2026-11-18 09:21:53,2026-11-18
2024-01-03T03:53:00Z,001i2LlKYto8Hja,enterprise,50,bb5c0c710b807ea6c3a62ff0,EU,active,Crescent Labs US,null,2026-11-18 09:21:53,2026-11-18
2024-01-03T03:53:00Z,001i2LlKYto8Hja,enterprise,50,bb5c0c710b807ea6c3a62ff0,EU,active,Crescent Labs US,null,2026-11-18 09:21:53,2026-11-18


DataFrame[created_at: string, crm_ref: string, license_plan: string, license_seats: string, org_id: string, region: string, status: string, tenant_name: string, license_addon: string, Ingestion_timestamp: string, Ingestion_date: date]

In [0]:
%%sql
select * from coreview.bronze.signins
order by event_ts, event_id

event_id,org_id,user_id,event_ts,result,ip_hash,Ingestion_timestamp,Ingestion_date
33e12ee093539e379842eda18e7d9fac,c36861da1967072fb7100ba6,272096371941e9ef,2026-01-01 00:00:34,success,520a6b9a3661161e,2026-11-18 11:34:25,2026-11-18
551074489f69c24a737d793d0d8b9e2d,c1afc50b65eca53e8b028074,b37de3be47746d22,2026-01-01 00:04:29,success,9853fad702c58c42,2026-11-18 11:34:25,2026-11-18
1a5ba64b08ae4d9097ec2f108b6c98dd,4ed9829119ecb3dbaa1570b8,9bd3d44b54dfece5,2026-01-01 00:10:30,success,7a532663d41f07e6,2026-11-18 11:34:25,2026-11-18
d547430ebb112a5566ea4badee68b73c,119c19de3be9772f43c8e798,ed3b23313b48b635,2026-01-01 00:15:12,success,4e6d620c7774c59c,2026-11-18 11:34:25,2026-11-18
1db9954f3d0377d0c4f681948700b2aa,58cfe411499a3d1dd16f9015,5960138bd09ef56c,2026-01-01 00:54:09,success,4ce519ac7d9bb2a0,2026-11-18 11:34:25,2026-11-18
6c31cbefe3246a516bf1b467ca667f63,a24be4ed752397b577e6c463,bc4279d490fca4c6,2026-01-01 00:55:38,success,433b38bb277762a7,2026-11-18 11:34:25,2026-11-18
a60da1aacbe9b341e94e49146d5ff191,a295c8292d93080960c57067,3549fcf2bb7c6698,2026-01-01 01:21:10,success,64bf2fa69131b7c9,2026-11-18 11:34:25,2026-11-18
7c8ecf5bab3c56ce6e6a4c32552abe64,77962f9bf1aa40abfd99e240,88ad3a81fa484616,2026-01-01 01:59:06,success,43f7e1fe85a41401,2026-11-18 11:34:25,2026-11-18
c554007e855019aebcdad501bfa67ca8,64a0089a1e781950c0df5de8,c098445ffc6b0b78,2026-01-01 02:12:43,success,2927714c8a080588,2026-11-18 11:34:25,2026-11-18
b5898773267824e2fc3c7f3d9a2dc8f7,1f380cb17357fa5aa402b2dd,7d221a7ab7a0513c,2026-01-01 02:24:41,success,8de027d22a643628,2026-11-18 11:34:25,2026-11-18


DataFrame[event_id: string, org_id: string, user_id: string, event_ts: string, result: string, ip_hash: string, Ingestion_timestamp: string, Ingestion_date: date]